# Lecture 6: Convolutional Neural Networks (CNN) and Residual Networks


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

np.random.seed(42)
print("Libraries loaded.")


## 1. 1D Convolution (Ch. 10.2)

Convolution slides a **filter (kernel)** over the input.  
Used in signal processing for smoothing, edge detection.

$$[f * g](t) = \\sum_\\tau f(\\tau) \\cdot g(t - \\tau)$$


In [ ]:
# 1D convolution example
np.random.seed(5)
signal = np.sin(np.linspace(0, 4*np.pi, 100)) + 0.3*np.random.randn(100)

kernels = {
    "Average (smoothing) [1/5,1/5,1/5,1/5,1/5]": np.ones(5)/5,
    "Edge detection [-1,0,1]": np.array([-1, 0, 1]),
    "Sharpen [-1,3,-1]": np.array([-1, 3, -1]),
}

fig, axes = plt.subplots(len(kernels), 1, figsize=(13, 9))
for ax, (name, kernel) in zip(axes, kernels.items()):
    filtered = np.convolve(signal, kernel, mode='same')
    ax.plot(signal, 'gray', lw=1, alpha=0.6, label="Original signal")
    ax.plot(filtered, 'crimson', lw=2, label=f"Filter: {name}")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle("1D Convolution: Different Filters", fontsize=14)
plt.tight_layout()
plt.savefig("fig_nb06_conv1d.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. 2D Convolution (Ch. 10.3)

2D convolution in image processing:
- **Edge detection**: Sobel, Laplacian filters
- **Blurring**: Gaussian filter
- **Sharpening**: Unsharp mask

In CNNs these filters are **learned** (not fixed).


In [ ]:
# Generate synthetic image
from skimage.draw import rectangle, disk

img = np.zeros((64, 64))
rr, cc = rectangle(start=(10, 10), end=(30, 40), shape=img.shape)
img[rr, cc] = 1.0
rr2, cc2 = disk((45, 45), 10, shape=img.shape)
img[rr2, cc2] = 0.7

kernels_2d = {
    "Sobel-X": np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    "Sobel-Y": np.array([[-1,-2,-1],[0,0,0],[1,2,1]]),
    "Laplacian": np.array([[0,1,0],[1,-4,1],[0,1,0]]),
    "Gaussian": np.array([[1,2,1],[2,4,2],[1,2,1]])/16,
}

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title("Original"); axes[0].axis('off')

for ax, (name, k) in zip(axes[1:], kernels_2d.items()):
    filtered = convolve2d(img, k, mode='same', boundary='symm')
    ax.imshow(np.abs(filtered), cmap='hot'); ax.set_title(name); ax.axis('off')

plt.suptitle("2D Convolution: Different Filters", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb06_conv2d.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. Padding and Stride

- **Padding**: Add border zeros to preserve output size  
- **Stride**: Step size → large stride = small output, computation savings

**Output size formula:**
$$W_{out} = \\lfloor\\frac{W_{in} + 2P - K}{S}\\rfloor + 1$$


In [ ]:
def manual_conv2d(x, kernel, padding=0, stride=1):
    if padding > 0:
        x = np.pad(x, padding, mode='constant')
    kH, kW = kernel.shape
    H, W = x.shape
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    out = np.zeros((out_H, out_W))
    for i in range(0, out_H):
        for j in range(0, out_W):
            patch = x[i*stride:i*stride+kH, j*stride:j*stride+kW]
            out[i, j] = (patch * kernel).sum()
    return out

edge_kernel = np.array([[-1,0,1],[-2,0,2],[-1,0,1]])

configs = [
    ("Padding=0, Stride=1", 0, 1),
    ("Padding=1, Stride=1", 1, 1),
    ("Padding=0, Stride=2", 0, 2),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title(f"Original ({img.shape[0]}x{img.shape[1]})"); axes[0].axis('off')

for ax, (name, pad, strd) in zip(axes[1:], configs):
    result = manual_conv2d(img, edge_kernel, padding=pad, stride=strd)
    ax.imshow(np.abs(result), cmap='hot')
    ax.set_title(f"{name}\nOutput: {result.shape[0]}x{result.shape[1]}")
    ax.axis('off')

plt.suptitle("Effect of Padding and Stride", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb06_padding_stride.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Pooling

- **Max Pooling**: Take the maximum value in the window → translation invariance
- **Average Pooling**: Take the mean of the window → soft downsampling


In [ ]:
def max_pool2d(x, pool_size=2, stride=2):
    H, W = x.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    out = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            patch = x[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            out[i, j] = patch.max()
    return out

def avg_pool2d(x, pool_size=2, stride=2):
    H, W = x.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    out = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            patch = x[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            out[i, j] = patch.mean()
    return out

feature_map = convolve2d(img, np.array([[-1,0,1],[-2,0,2],[-1,0,1]]), mode='same', boundary='symm')
feature_abs = np.abs(feature_map)

mp = max_pool2d(feature_abs, pool_size=2, stride=2)
ap = avg_pool2d(feature_abs, pool_size=2, stride=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(feature_abs, cmap='hot'); axes[0].set_title(f"Feature Map ({feature_abs.shape[0]}x{feature_abs.shape[1]})"); axes[0].axis('off')
axes[1].imshow(mp, cmap='hot'); axes[1].set_title(f"Max Pool 2x2 ({mp.shape[0]}x{mp.shape[1]})"); axes[1].axis('off')
axes[2].imshow(ap, cmap='hot'); axes[2].set_title(f"Avg Pool 2x2 ({ap.shape[0]}x{ap.shape[1]})"); axes[2].axis('off')

plt.suptitle("Pooling: Spatial Dimensionality Reduction of Feature Maps", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb06_pooling.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. CNN Architecture (LeNet-like)

Typical CNN pipeline:
```
Input → [Conv → ReLU → Pool] × N → Flatten → FC → FC → Output
```

Each conv-pool block reduces spatial size and increases the number of channels.


In [ ]:
def cnn_layer_info(in_h, in_w, in_c, out_c, kernel=3, padding=1, stride=1, pool=2):
    conv_h = (in_h + 2*padding - kernel) // stride + 1
    conv_w = (in_w + 2*padding - kernel) // stride + 1
    pool_h = conv_h // pool
    pool_w = conv_w // pool
    params = (kernel*kernel*in_c + 1) * out_c
    return conv_h, conv_w, pool_h, pool_w, params

# LeNet-like architecture
layers = [("Input", 32, 32, 1, "—", "—")]

h, w, c = 32, 32, 1
configs = [(6, 5, 0, 2), (16, 5, 0, 2)]
for out_c, k, pad, pool in configs:
    ch, cw, ph, pw, params = cnn_layer_info(h, w, c, out_c, k, pad, 1, pool)
    layers.append((f"Conv({k}x{k})+ReLU", ch, cw, out_c, params, "—"))
    layers.append((f"MaxPool({pool}x{pool})", ph, pw, out_c, "—", "—"))
    h, w, c = ph, pw, out_c

flat = h * w * c
layers.append((f"Flatten", flat, 1, 1, "—", "—"))
layers.append(("FC-120", 120, 1, 1, flat*120+120, "—"))
layers.append(("FC-84",   84, 1, 1, 120*84+84, "—"))
layers.append(("FC-10",   10, 1, 1, 84*10+10, "Softmax"))

print(f"{'Layer':<25} {'Size (H)':<10} {'Channels':<8} {'Parameters'}")
print("-"*60)
for name, H, W, C, params, act in layers:
    print(f"{name:<25} {str(H)+('x'+str(W) if W>1 else ''):<10} {C:<8} {params}")


## 6. Residual Connections (Ch. 11)

To solve the vanishing gradient problem in deep networks:

$$\\mathbf{h}_{l+1} = F(\\mathbf{h}_l) + \\mathbf{h}_l$$

The learned quantity `F(x)` is the **residual** function, starting from zero rather than an arbitrary initialization.  
Gradients flow directly through the skip connection → very deep networks can be trained.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Gradient flow: with vs without skip connection
def simulate_gradient_flow(n_layers=20, has_skip=False):
    np.random.seed(1)
    grads = []
    g = 1.0
    for i in range(n_layers):
        layer_factor = np.random.uniform(0.5, 0.9)
        if has_skip:
            g = 0.5 * g * layer_factor + 0.5 * 1.0
        else:
            g = g * layer_factor
        grads.append(g)
    return grads

grads_no_skip   = simulate_gradient_flow(n_layers=20, has_skip=False)
grads_with_skip = simulate_gradient_flow(n_layers=20, has_skip=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(grads_no_skip,   color='crimson',   marker='o', markersize=5, lw=2, label="No skip connection")
axes[0].plot(grads_with_skip, color='steelblue', marker='s', markersize=5, lw=2, label="With skip connection")
axes[0].set_title("Gradient Magnitude per Layer")
axes[0].set_xlabel("Layer (backward)"); axes[0].set_ylabel("Gradient magnitude")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_yscale('log')

ax = axes[1]; ax.set_xlim(0,10); ax.set_ylim(0,8); ax.axis('off')
ax.set_title("ResNet Block Structure")

def draw_box(ax, x, y, w, h, text, color='steelblue'):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=9, color='white', fontweight='bold')

draw_box(ax, 3.5, 6,   3, 0.8, "x (input)", 'gray')
draw_box(ax, 3.5, 4.5, 3, 0.8, "Weight Layer", 'steelblue')
draw_box(ax, 3.5, 3.2, 3, 0.8, "Batch Norm + ReLU", 'darkorange')
draw_box(ax, 3.5, 1.9, 3, 0.8, "Weight Layer", 'steelblue')
draw_box(ax, 3.5, 0.6, 3, 0.8, "x + F(x) (output)", 'limegreen')

for (x1,y1),(x2,y2) in [((5,6),(5,5.3)), ((5,5.3),(5,4.0)), ((5,4.0),(5,2.7)), ((5,2.7),(5,1.4))]:
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->", color='white', lw=2))

ax.annotate("", xy=(7.5, 1.0), xytext=(7.5, 6.4),
            arrowprops=dict(arrowstyle="->", color='limegreen', lw=2.5,
                           connectionstyle="arc3,rad=-0.3"))
ax.text(8.5, 3.5, "Skip\nConnection\nF(x)+x", ha='center', fontsize=9, color='limegreen')

plt.tight_layout()
plt.savefig("fig_nb06_resnet.png", dpi=100, bbox_inches='tight')
plt.show()


## 7. Summary

| Concept | Description |
|---|---|
| **Convolution** | Sliding filter application — learns local patterns |
| **Padding** | Preserves output size — prevents information loss at borders |
| **Stride** | Computational savings, spatial downsampling |
| **Max Pooling** | Translation invariance, dimensionality reduction |
| **Skip Connection** | Prevents vanishing gradients, enables training of deep networks |
| **Batch Norm** | Normalizes activations in each block |

**Next Notebook →** Transformers (Attention Mechanism)
